# Deepfake Detection Experiments

This notebook contains end-to-end experiments for:
- Frame-level model training using `data/img dataset`
- Image prediction sanity checks
- Video-level deepfake detection by aggregating frame predictions

Final output decision is interpreted as either:
- `real`
- `deepfake_or_ai_generated`

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Make project root importable when running notebook from notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("CUDA available:", __import__("torch").cuda.is_available())

In [ ]:
# Paths and experiment settings
DATA_DIR = PROJECT_ROOT / "data" / "img dataset"
MODEL_PATH = PROJECT_ROOT / "models" / "video_detection_frame_model.pth"

EPOCHS = 10
BATCH_SIZE = 32
IMAGE_SIZE = 224
LEARNING_RATE = 1e-3
NUM_WORKERS = 2
MAX_VIDEO_FRAMES = 32

print("DATA_DIR:", DATA_DIR)
print("MODEL_PATH:", MODEL_PATH)

In [ ]:
# Dataset validation and quick class distribution check
from collections import Counter

from src.image_detection.data_ingestion import get_dataset_paths

dataset_paths = get_dataset_paths(str(DATA_DIR))
print(dataset_paths)

for split_name, split_dir in {
    "train": dataset_paths.train_dir,
    "val": dataset_paths.val_dir,
    "test": dataset_paths.test_dir,
}.items():
    counts = Counter()
    for class_dir in split_dir.iterdir():
        if class_dir.is_dir():
            counts[class_dir.name] = len(list(class_dir.glob("*")))
    print(f"\n{split_name.upper()} distribution:")
    for k, v in sorted(counts.items()):
        print(f"  {k}: {v}")

In [ ]:
# Build dataloaders for training/validation/testing
from src.image_detection.data_transformation import build_dataloaders

dataloaders = build_dataloaders(
    dataset_paths=dataset_paths,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

print("Classes:", dataloaders.class_names)
print("Train batches:", len(dataloaders.train))
print("Val batches:", len(dataloaders.val))
print("Test batches:", len(dataloaders.test))

In [ ]:
# Train frame-level model (used later for video prediction)
from src.image_detection.model_evaluation import evaluate_model
from src.image_detection.model_trainer import TrainConfig, train_model

train_config = TrainConfig(
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    model_path=str(MODEL_PATH),
    freeze_backbone=True,
)

model, train_metrics = train_model(
    train_loader=dataloaders.train,
    val_loader=dataloaders.val,
    num_classes=len(dataloaders.class_names),
    config=train_config,
)

test_metrics = evaluate_model(model, dataloaders.test)
print("Train metrics:", train_metrics)
print("Test metrics:", test_metrics)

In [ ]:
# Save class names alongside model for stable inference
classes_file = MODEL_PATH.with_suffix(".classes.txt")
classes_file.parent.mkdir(parents=True, exist_ok=True)
classes_file.write_text("\n".join(dataloaders.class_names), encoding="utf-8")
print("Saved class names to:", classes_file)
print(classes_file.read_text(encoding="utf-8"))

In [ ]:
# Image-level sanity check on one sample frame
from src.image_detection.predict import predict_image

sample_image = None
for class_name in dataloaders.class_names:
    candidates = list((dataset_paths.test_dir / class_name).glob("*.png"))
    if candidates:
        sample_image = candidates[0]
        break

if sample_image is None:
    raise FileNotFoundError("No sample image found in test split.")

label, confidence = predict_image(
    image_path=str(sample_image),
    model_path=str(MODEL_PATH),
    class_names=dataloaders.class_names,
    image_size=IMAGE_SIZE,
)

print("Sample image:", sample_image)
print("Prediction:", label)
print("Confidence:", f"{confidence:.2%}")

In [ ]:
# Video-level prediction experiment
# Place test videos inside: <project_root>/data/videos
from src.video_detection.video_classifier import predict_video

videos_dir = PROJECT_ROOT / "data" / "videos"
video_files = [
    *videos_dir.glob("*.mp4"),
    *videos_dir.glob("*.avi"),
    *videos_dir.glob("*.mov"),
    *videos_dir.glob("*.mkv"),
]

if not video_files:
    raise FileNotFoundError(
        f"No videos found in {videos_dir}. Add at least one video and rerun this cell."
    )

video_path = video_files[0]
result = predict_video(
    video_path=str(video_path),
    model_path=str(MODEL_PATH),
    class_names=dataloaders.class_names,
    image_size=IMAGE_SIZE,
    max_frames=MAX_VIDEO_FRAMES,
)

print("Video:", video_path)
print("Predicted class:", result.predicted_label)
print("Confidence:", f"{result.confidence:.2%}")
print("Final decision:", result.decision)
print("Frames used:", result.total_frames_used)
print("Class probabilities:")
for class_name, score in result.class_probabilities.items():
    print(f"  {class_name}: {score:.2%}")

In [ ]:
# Batch video experiment (evaluate multiple videos quickly)
results = []
for video_path in video_files:
    pred = predict_video(
        video_path=str(video_path),
        model_path=str(MODEL_PATH),
        class_names=dataloaders.class_names,
        image_size=IMAGE_SIZE,
        max_frames=MAX_VIDEO_FRAMES,
    )
    results.append(
        {
            "video": video_path.name,
            "predicted_class": pred.predicted_label,
            "confidence": round(pred.confidence, 4),
            "decision": pred.decision,
            "frames_used": pred.total_frames_used,
        }
    )

results